# Xử lý Missing Values - Hotel Room Price Dataset

Notebook này sẽ:
1. Kiểm tra missing values trong dataset
2. Phân tích tính khả thi của Stochastic KNN Imputation
3. Triển khai Stochastic KNN Imputation nếu khả thi


In [29]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# For KNN Imputation
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

%matplotlib inline


In [30]:
# Load dataset
df = pd.read_csv('final_selected_data.csv', low_memory=False)
print(f"Dataset shape: {df.shape}")
print(f"Số dòng: {len(df):,}")
print(f"Số cột: {len(df.columns)}")


Dataset shape: (16509, 40)
Số dòng: 16,509
Số cột: 40


## 1. Kiểm tra Missing Values


In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

# Load CSV file from local directory
# Change the file name below to match your CSV file name
file_name = 'final_selected_data.csv'

# Load the data into a pandas DataFrame
try:
    df = pd.read_csv(file_name, low_memory=False)
    print(f"✓ Successfully loaded '{file_name}' into a DataFrame.")
    print(f"  - Shape: {df.shape}")
except FileNotFoundError:
    print(f"❌ Error: File '{file_name}' not found!")
    print(f"   Please make sure the CSV file is in the same directory as this notebook.")
    print(f"   Or update the 'file_name' variable above with the correct path.")
except Exception as e:
    print(f"❌ Error loading CSV: {e}")

# Define boolean columns that need to be converted to binary (0/1) first
# These columns have True/NaN values and should be converted to 1/0
# NOTE: 'sqm' and 'views' are NOT included here - they will remain unchanged
# - 'sqm' is numerical (float64) and will stay as numerical
# - 'views' is categorical (text) and will stay as categorical
boolean_to_binary_cols = ['balcony-terrace', 'closet', 'air_conditioning', 'mini_bar', 
                          'bathtub', 'shower', 'refrigerator', 'private-pool', 'jacuzzi-bathtub']

print("\n--- Step 1: Convert Boolean Columns to Binary (0/1) ---")
# Convert boolean columns to binary before processing
for col in boolean_to_binary_cols:
    if col in df.columns:
        # Convert True -> 1, False/NaN -> 0
        # df[col] = df[col].notna().astype(int)
        df[col] = df[col].notna().fillna(False).astype(int)
        print(f"Converted '{col}' to binary (0/1).")
    else:
        print(f"Warning: Boolean column '{col}' not found in DataFrame.")

# Define categorical columns (excluding boolean columns that are now binary)
categorical_cols = ['views', 'room_room_type_name', 'hotel_name', 'region',
                    'large_double_bed', 'large_bed', 'single_bed', 'sofa_bed', 
                    'double_bed', 'small_double_bed', 'king_size_bed', 'futon_mattress', 
                    'bunk_bed', 'extra_long_bed', 'bãi_đậu_xe', 'phòng_tập', 
                    'vào_hồ_bơi_miễn_phí', 'wifi_miễn_phí', 'không_hoàn_tiền', 
                    'miễn_phí_hủy', 'đã_kèm_bữa_sáng']
# Add the converted binary columns to categorical (they are now 0/1 but represent categories)
categorical_cols.extend(boolean_to_binary_cols)

# Define numerical columns
numerical_cols = ['room_type_id', 'flexibility_score', 'sqm', 'bathroom_count', 
                  'bedroom_count', 'price_option_price', 'adults_number', 
                  'children_number', 'hotel_id', 'có_bữa_sáng']

print("\n--- Step 2: Convert Categorical Columns to 'category' dtype ---")
# Convert categorical columns to 'category' dtype
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')
        print(f"Converted '{col}' to category type.")
    else:
        print(f"Warning: Categorical column '{col}' not found in DataFrame.")

print("\n--- Step 3: Convert Numerical Columns to Numeric Type ---")
# Attempt to convert numerical columns to a suitable numerical type, handling errors
for col in numerical_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"Converted '{col}' to numeric type (coerced errors).")
    else:
        print(f"Warning: Numerical column '{col}' not found in DataFrame.")

print("\n--- Identified Column Types ---")
print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

print("\n--- Step 4: Check Missing Values in Target Variable (price_option_price) ---")
# Check for missing values in the target variable
missing_price_count = df['price_option_price'].isna().sum()
total_rows = len(df)
missing_percentage = (missing_price_count / total_rows) * 100

print(f"Missing values in 'price_option_price': {missing_price_count} out of {total_rows} ({missing_percentage:.2f}%)")

if missing_price_count > 0:
    print("\n⚠️  WARNING: Target variable has missing values!")
    print("\nAnalysis of missing price_option_price:")
    missing_df = df[df['price_option_price'].isna()]
    print(f"\nHotels with missing prices:")
    print(missing_df['hotel_name'].value_counts().head(10))
    print(f"\nRoom types with missing prices:")
    print(missing_df['room_room_type_name'].value_counts().head(10))
    if 'region' in df.columns:
        print(f"\nRegions with missing prices:")
        print(missing_df['region'].value_counts())
    
    print("\n💡 RECOMMENDATION:")
    print("Since price_option_price is the target variable, you have two options:")
    print("1. Drop rows with missing prices (if acceptable for your analysis)")
    print("2. Impute missing prices (e.g., using median by hotel/region/room_type)")
    print("\nTo drop missing values, uncomment the line below:")
    print("# df = df.dropna(subset=['price_option_price'])")
else:
    print("✓ No missing values in target variable!")

print("\n--- DataFrame Info After Conversions ---")
df.info()

✓ Successfully loaded 'final_selected_data.csv' into a DataFrame.
  - Shape: (16509, 40)

--- Step 1: Convert Boolean Columns to Binary (0/1) ---
Converted 'balcony-terrace' to binary (0/1).
Converted 'closet' to binary (0/1).
Converted 'air_conditioning' to binary (0/1).
Converted 'mini_bar' to binary (0/1).
Converted 'bathtub' to binary (0/1).
Converted 'shower' to binary (0/1).
Converted 'refrigerator' to binary (0/1).
Converted 'private-pool' to binary (0/1).
Converted 'jacuzzi-bathtub' to binary (0/1).

--- Step 2: Convert Categorical Columns to 'category' dtype ---
Converted 'views' to category type.
Converted 'room_room_type_name' to category type.
Converted 'hotel_name' to category type.
Converted 'region' to category type.
Converted 'large_double_bed' to category type.
Converted 'large_bed' to category type.
Converted 'single_bed' to category type.
Converted 'sofa_bed' to category type.
Converted 'double_bed' to category type.
Converted 'small_double_bed' to category type.
Con

In [32]:
print("--- DataFrame Shape ---")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\n--- DataFrame Info ---")
df.info()

print("\n--- Descriptive Statistics ---")
display(df.describe())

print("\n--- Data Types of Each Column ---")
print(df.dtypes)

print("\n--- First 5 Rows of DataFrame ---")
display(df.head())

print("\n--- Column Names List ---")
print(df.columns.tolist())

print("\n--- Number of Missing Values Per Column ---")
print(df.isnull().sum())

--- DataFrame Shape ---
Rows: 16509
Columns: 40

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16509 entries, 0 to 16508
Data columns (total 40 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   room_type_id         16509 non-null  int64   
 1   large_double_bed     16509 non-null  category
 2   large_bed            16509 non-null  category
 3   single_bed           16509 non-null  category
 4   sofa_bed             16509 non-null  category
 5   double_bed           16509 non-null  category
 6   small_double_bed     16509 non-null  category
 7   king_size_bed        16509 non-null  category
 8   futon_mattress       16509 non-null  category
 9   bunk_bed             16509 non-null  category
 10  extra_long_bed       16509 non-null  category
 11  flexibility_score    16509 non-null  int64   
 12  sqm                  15169 non-null  float64 
 13  views                12265 non-null  category
 14

,room_type_id,flexibility_score,sqm,bathroom_count,bedroom_count,price_option_price,adults_number,children_number,có_bữa_sáng,hotel_id
count,16509.000000,16509.000000,15169.000000,16509.000000,16509.000000,1.647900e+04,16509.000000,16509.000000,16509.000000,1.650900e+04
mean,5123.554788,1.050457,40.170941,0.967472,0.252589,1.376844e+06,2.493488,0.465988,104735.337392,3.670931e+07
std,2875.313213,0.222462,88.482364,0.982015,0.970634,5.903493e+06,1.807835,0.744097,39278.638644,2.746972e+07
min,1.000000,1.000000,4.000000,0.000000,0.000000,3.070000e+02,0.000000,0.000000,3.000000,1.094300e+04
25%,2714.000000,1.000000,22.000000,1.000000,0.000000,4.125000e+05,2.000000,0.000000,100000.000000,6.908956e+06
50%,5131.000000,1.000000,30.000000,1.000000,0.000000,6.747690e+05,2.000000,0.000000,100000.000000,3.700605e+07
75%,7652.000000,1.000000,40.000000,1.000000,0.000000,1.245896e+06,2.000000,1.000000,100000.000000,6.253754e+07
max,10066.000000,3.000000,6000.000000,21.000000,21.000000,6.666667e+08,50.000000,12.000000,759000.000000,8.105244e+07



--- Data Types of Each Column ---
room_type_id              int64
large_double_bed       category
large_bed              category
single_bed             category
sofa_bed               category
double_bed             category
small_double_bed       category
king_size_bed          category
futon_mattress         category
bunk_bed               category
extra_long_bed         category
flexibility_score         int64
sqm                     float64
views                  category
bathroom_count            int64
balcony-terrace        category
closet                 category
air_conditioning       category
mini_bar               category
bathtub                category
shower                 category
refrigerator           category
private-pool           category
jacuzzi-bathtub        category
bedroom_count             int64
price_option_price      float64
adults_number             int64
children_number           int64
có_bữa_sáng               int64
bãi_đậu_xe             category
phòng

,room_type_id,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,...,phòng_tập,vào_hồ_bơi_miễn_phí,wifi_miễn_phí,không_hoàn_tiền,miễn_phí_hủy,đã_kèm_bữa_sáng,hotel_id,room_room_type_name,hotel_name,region
0,1,1,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,71897952,Phòng Deluxe Có Giường Cỡ King (Deluxe King Room),Nha Nghi Nhung - Nhung Motel,Bà Rịa
1,2,0,1,0,0,0,0,0,0,0,...,0,0,1,0,1,0,49685029,Phòng Tiêu Chuẩn (Standard Room),Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa),Bà Rịa
2,3,0,1,0,0,0,0,0,0,0,...,0,0,1,0,1,0,49685029,Phòng gia đình có ban công (Family Room with B...,Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa),Bà Rịa
3,4,0,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,65481766,Phòng Có Giường Cỡ King Với Ban Công (King Roo...,Baly Hotel Bà Rịa City (Baly Hotel Ba Ria City),Bà Rịa
4,5,1,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,5808626,Phòng Studio Executive (Studio Executive),Citadines Central Bình Dương (Citadines Centra...,Bình Dương



--- Column Names List ---
['room_type_id', 'large_double_bed', 'large_bed', 'single_bed', 'sofa_bed', 'double_bed', 'small_double_bed', 'king_size_bed', 'futon_mattress', 'bunk_bed', 'extra_long_bed', 'flexibility_score', 'sqm', 'views', 'bathroom_count', 'balcony-terrace', 'closet', 'air_conditioning', 'mini_bar', 'bathtub', 'shower', 'refrigerator', 'private-pool', 'jacuzzi-bathtub', 'bedroom_count', 'price_option_price', 'adults_number', 'children_number', 'có_bữa_sáng', 'bãi_đậu_xe', 'phòng_tập', 'vào_hồ_bơi_miễn_phí', 'wifi_miễn_phí', 'không_hoàn_tiền', 'miễn_phí_hủy', 'đã_kèm_bữa_sáng', 'hotel_id', 'room_room_type_name', 'hotel_name', 'region']

--- Number of Missing Values Per Column ---
room_type_id              0
large_double_bed          0
large_bed                 0
single_bed                0
sofa_bed                  0
double_bed                0
small_double_bed          0
king_size_bed             0
futon_mattress            0
bunk_bed                  0
extra_long_bed 

## 2. Phân tích Missing Values và Chuẩn bị cho KNN Imputation


In [33]:
# Phân tích missing values chi tiết
print("=" * 70)
print("PHÂN TÍCH MISSING VALUES CHI TIẾT")
print("=" * 70)

missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Missing Percent (%)': missing_percent.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print("\nCác cột có missing values:")
print(missing_df.to_string(index=False))

print(f"\nTổng số missing values: {missing_values.sum():,}")

# Phân loại các cột (định nghĩa lại để đảm bảo có sẵn)
numerical_cols = ['room_type_id', 'flexibility_score', 'sqm', 'bathroom_count', 
                  'bedroom_count', 'price_option_price', 'adults_number', 
                  'children_number', 'hotel_id', 'có_bữa_sáng']

# Binary columns (0/1) - đã được convert trong preprocessing
binary_cols = ['large_double_bed', 'large_bed', 'single_bed', 'sofa_bed', 
               'double_bed', 'small_double_bed', 'king_size_bed', 'futon_mattress', 
               'bunk_bed', 'extra_long_bed', 'bãi_đậu_xe', 'phòng_tập', 
               'vào_hồ_bơi_miễn_phí', 'wifi_miễn_phí', 'không_hoàn_tiền', 
               'miễn_phí_hủy', 'đã_kèm_bữa_sáng']

# Boolean columns (đã convert thành 0/1 trong preprocessing)
boolean_cols = ['balcony-terrace', 'closet', 'air_conditioning', 'mini_bar', 
                'bathtub', 'shower', 'refrigerator', 'private-pool', 'jacuzzi-bathtub']
binary_cols.extend(boolean_cols)

# Categorical columns với nhiều giá trị (CHỈ còn views - sẽ impute)
categorical_cols = ['views']

# Các cột không impute (giữ nguyên missing values)
cols_not_to_impute_list = ['hotel_name', 'region', 'room_room_type_name']

print("\n" + "=" * 70)
print("PHÂN LOẠI CÁC CỘT")
print("=" * 70)
print(f"Numerical columns: {len(numerical_cols)}")
print(f"Binary columns: {len(binary_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Boolean columns (đã convert): {len(boolean_cols)}")

# Kiểm tra các cột có missing values theo loại
cols_with_missing = missing_df['Column'].tolist()
print(f"\nCác cột có missing values:")
print(f"  - Numerical: {[c for c in cols_with_missing if c in numerical_cols]}")
print(f"  - Categorical (sẽ impute): {[c for c in cols_with_missing if c in categorical_cols]}")
print(f"  - Boolean: {[c for c in cols_with_missing if c in boolean_cols]}")
print(f"  - Không impute (giữ nguyên): {[c for c in cols_with_missing if c in cols_not_to_impute_list]}")


PHÂN TÍCH MISSING VALUES CHI TIẾT

Các cột có missing values:
            Column  Missing Count  Missing Percent (%)
             views           4244            25.707190
               sqm           1340             8.116785
        hotel_name            240             1.453752
            region            240             1.453752
price_option_price             30             0.181719

Tổng số missing values: 6,094

PHÂN LOẠI CÁC CỘT
Numerical columns: 10
Binary columns: 26
Categorical columns: 1
Boolean columns (đã convert): 9

Các cột có missing values:
  - Numerical: ['sqm', 'price_option_price']
  - Categorical (sẽ impute): ['views']
  - Boolean: []
  - Không impute (giữ nguyên): ['hotel_name', 'region']


## 3. Stochastic KNN Imputation
### Step 1: Chuẩn bị dữ liệu cho distance

- Numerical: scale về [0,1]
- Binary: giữ nguyên 0/1
- Categorical (views): giữ dạng string, không encode views

In [34]:
from sklearn.preprocessing import MinMaxScaler

df_work = df.copy()

# Scale numerical columns
scaler = MinMaxScaler()
df_work[numerical_cols] = scaler.fit_transform(df_work[numerical_cols])


### Step 2: Hàm tính Gower distance

In [35]:
import numpy as np

def gower_distance(row, df, numerical_cols, binary_cols, categorical_cols):
    distances = []

    # Numerical
    for col in numerical_cols:
        if pd.notna(row[col]):
            diff = np.abs(df[col] - row[col])
            distances.append(diff)

    # Binary
    for col in binary_cols:
        if pd.notna(row[col]):
            diff = (df[col] != row[col]).astype(int)
            distances.append(diff)

    # Categorical
    for col in categorical_cols:
        if pd.notna(row[col]):
            diff = (df[col] != row[col]).astype(int)
            distances.append(diff)

    if not distances:
        return pd.Series(np.inf, index=df.index)

    return sum(distances) / len(distances)


### Step 4: Stochastic KNN Imputation

In [36]:
import random

def stochastic_knn_impute(
    df,
    numerical_cols,
    binary_cols,
    categorical_cols,
    k=5,
    random_state=42
):
    np.random.seed(random_state)
    random.seed(random_state)

    df_imputed = df.copy()

    for idx, row in df[df.isnull().any(axis=1)].iterrows():

        # Tính distance đến các dòng KHÔNG missing toàn bộ
        candidate_df = df.drop(idx)

        distances = gower_distance(
            row,
            candidate_df,
            numerical_cols,
            binary_cols,
            categorical_cols
        )

        nearest_idx = distances.nsmallest(k).index
        neighbors = candidate_df.loc[nearest_idx]

        # ===== Numerical =====
        for col in numerical_cols:
            if pd.isna(row[col]):
                vals = neighbors[col].dropna()
                if len(vals) > 0:
                    df_imputed.at[idx, col] = np.random.uniform(
                        vals.min(), vals.max()
                    )

        # ===== Binary (0/1) =====
        for col in binary_cols:
            if pd.isna(row[col]):
                vals = neighbors[col].dropna()
                if len(vals) > 0:
                    p = vals.mean()
                    df_imputed.at[idx, col] = np.random.binomial(1, p)

        # ===== Categorical (views) =====
        for col in categorical_cols:
            if pd.isna(row[col]):
                vals = neighbors[col].dropna()
                if len(vals) > 0:
                    df_imputed.at[idx, col] = np.random.choice(vals.values)

    return df_imputed


### Step 5: Chạy Imputation và Scale numerical back

In [37]:
df_imputed = stochastic_knn_impute(
    df_work,
    numerical_cols=numerical_cols,
    binary_cols=binary_cols,
    categorical_cols=categorical_cols,
    k=7
)

In [38]:
df_imputed[numerical_cols] = scaler.inverse_transform(
    df_imputed[numerical_cols]
)


In [39]:
print("--- DataFrame Shape ---")
print("Rows:", df_imputed.shape[0])
print("Columns:", df_imputed.shape[1])

print("\n--- Number of Missing Values Per Column ---")
print(df_imputed.isnull().sum())

--- DataFrame Shape ---
Rows: 16509
Columns: 40

--- Number of Missing Values Per Column ---
room_type_id             0
large_double_bed         0
large_bed                0
single_bed               0
sofa_bed                 0
double_bed               0
small_double_bed         0
king_size_bed            0
futon_mattress           0
bunk_bed                 0
extra_long_bed           0
flexibility_score        0
sqm                    159
views                  848
bathroom_count           0
balcony-terrace          0
closet                   0
air_conditioning         0
mini_bar                 0
bathtub                  0
shower                   0
refrigerator             0
private-pool             0
jacuzzi-bathtub          0
bedroom_count            0
price_option_price       0
adults_number            0
children_number          0
có_bữa_sáng              0
bãi_đậu_xe               0
phòng_tập                0
vào_hồ_bơi_miễn_phí      0
wifi_miễn_phí            0
không_hoàn_tiền 

In [40]:
# Amenities Analysis - Impact on Price

amenity_cols = [
    col for col in categorical_cols
    if col in df.columns
    and col not in ['region', 'hotel_name', 'room_room_type_name', 'views']
    and set(df[col].dropna().unique()).issubset({0, 1})
]
# Filter to only include columns that exist
amenity_cols = [col for col in amenity_cols if col in df.columns]

if len(amenity_cols) > 0 and 'price_option_price' in df.columns:
    df_clean = df.dropna(subset=['price_option_price'])
    
    # Calculate average price with and without each amenity
    amenity_impact = {}
    for col in amenity_cols:
        if col in df_clean.columns:
            avg_with = df_clean[df_clean[col] == 1]['price_option_price'].mean()
            avg_without = df_clean[df_clean[col] == 0]['price_option_price'].mean()
            amenity_impact[col] = {
                'With': avg_with,
                'Without': avg_without,
                'Difference': avg_with - avg_without
            }
    
    if amenity_impact:
        impact_df = pd.DataFrame(amenity_impact).T.sort_values('Difference', ascending=False)
        
        plt.figure(figsize=(12, 6))
        impact_df['Difference'].plot(kind='barh', color='darkgreen')
        plt.title('Price Impact of Amenities (Average Price Difference: With vs Without)')
        plt.xlabel('Price Difference (VNĐ)')
        plt.ylabel('Amenity')
        plt.axvline(x=0, color='red', linestyle='--', alpha=0.5)
        # Format x-axis to avoid scientific notation
        plt.ticklabel_format(style='plain', axis='x')
        # Format x-axis labels with thousand separators
        ax = plt.gca()
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))
        plt.tight_layout()
        plt.show()
        
        print("\n--- Amenity Impact on Price ---")
        display(impact_df)


In [42]:
df_imputed["private-pool"].value_counts()

private-pool
0    15686
1      823
Name: count, dtype: int64

In [43]:
# Lưu dataset sau khi fill missing values
output_path = "data_after_fill_missing.csv"

df_imputed.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"✓ Đã lưu dữ liệu sau khi fill missing vào file: {output_path}")
print(f"✓ Số dòng: {df_imputed.shape[0]:,}")
print(f"✓ Số cột: {df_imputed.shape[1]}")


✓ Đã lưu dữ liệu sau khi fill missing vào file: data_after_fill_missing.csv
✓ Số dòng: 16,509
✓ Số cột: 40
